# Constructed seed sets on the redraws of B

`fresh_pair_scores.ipynb` ran every constructed set once, on its pair's saved B. Sets scored above every random set succeeded there 88–98% of the time, and the ten best sets found for the frozen score 229 of 238 times (96%). One run per set cannot say whether a given set is reliable or was lucky on that B. Here every constructed set meets the same 10 redraws of B as the 100 random sets of `b_replicates`, so each set gets a success rate over 11 Bs (the saved B is draw 0) with an interval.

**Sets, per pair** (chosen on A only, with the frozen score and the pair's original normalisation):
- **Every set already constructed:** metric extremes, intermediate sets, pool tails and rules from `selected_sets.csv` and `metric_sets.csv` (about 96 distinct sets per pair). Their saved-B runs are reused.
- **Lowest score found:** the 3 lowest-scoring sets found by swap search, below the random sets.
- **More intermediate sets:** swap paths run from 5 starts to 8 targets, one vertex at a time.
  - Starts: the random sets at the 5th, 35th, 65th and 95th percentiles of the pair's random scores, and the top-degree rule set.
  - Targets: the best set of each metric group, and the lowest-score set.
  - 40 sets are kept, 2 per equal-width score bin: 13 bins above the random range, 4 across it and 3 below it. Within a bin they are spread over targets.
  - They run on the saved B too.

**Draws of B.** `b_replicates`' 10 draws, regenerated from its seeds. Each draw's hash is checked against `b_replicates`, so the constructed and random sets meet exactly the same Bs.

**Size.** 3,749 sets (2,588 already run on the saved B, 1,161 new) and 38,651 runs: the new sets on the saved B, and every set on each of the 10 draws. Constructed sets converge faster than random ones, so this should take about 5–6 hours with 3 jobs (a rough guess from the saved-B run times; `b_replicates`' 27,000 random-set runs took 5.2 hours).

**Order.** Draw by draw: every set on every pair gets draw 1 before any set gets draw 2. If the run is stopped early, every set still has the same number of draws, and the tables below use only draws that every set of a pair has finished.

In [ ]:
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'experiment_common').is_dir())
if ROOT.name == 'selection_rule_comparison':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from selection_rule_comparison import constructed_redraws as CR
CR = importlib.reload(CR)   # picks up edits without restarting the kernel

JOBS = 3
BLAS_THREADS = 3   # keep 3: the saved-B runs and b_replicates used 3, and the thread count can change FAQ/GOAT outcomes
print('Results:', CR.OUTPUT_DIR)
display(CR.status())

## Run or resume

For an overnight run a terminal is safer than this notebook, because it survives a closed browser or a kernel restart. From the repository root, in the `ds` environment:

```bash
python -m selection_rule_comparison.constructed_redraws --jobs 3
python -m selection_rule_comparison.constructed_redraws --status    # from another window
```

The next cell does the same. The first minute builds the set lists, then runs start. Interrupting stops the workers and keeps every finished run; rerunning resumes. Keep `BLAS_THREADS` at 3 when resuming. To try another design, set `CONSTRUCTED_REDRAWS_OUTPUT_DIR` before importing the module.

In [ ]:
RUN = True
if RUN:
    CR.run_all(jobs=JOBS, blas_threads=BLAS_THREADS)
display(CR.status())

## Sets scored above every random set

This is the per-set version of the single-B table in `fresh-pair-extremes-findings`. It covers the constructed sets whose score is above the highest of their pair's 1,000 random scores, without the lowest-score sets.

- **saved B (one run):** the share that succeeded on the saved B, as in the earlier table.
- **success over all Bs:** pooled over every set and draw. The interval resamples pairs, then sets within pairs, so it allows for sets of a pair sharing their Bs.
- **median set, worst set:** the per-set success rates over the Bs.
- **sets ≥ 0.9, sets on every B:** shares of sets.
- **sets above random (Wilson):** the share of sets whose 95% lower bound is above the random sets' mean rate on the same pair and the same Bs. With 11 Bs, a set that succeeds on all of them has a lower bound of 0.74.
- **random sets, same pairs; top 10 random by score:** `b_replicates`' random sets on the same Bs: all 100 per pair, and the 10 highest-scoring.

In [ ]:
runs = CR.read_runs()             # only draws that every set of a pair has finished
random = CR.random_rates(runs)    # b_replicates' random sets on the same Bs
rates = CR.set_rates(runs, random=random)
print(f"{len(rates):,} constructed sets, {len(runs):,} runs; Bs per set:", sorted(int(d) for d in rates['draws'].unique()))
display(CR.above_random_table(rates, random))
display(CR.above_random_table(rates, random, by_model=True))

## The ten best sets found for the frozen score

On the saved B they succeeded 229 of 238 times (96%). The same sets over all Bs, per algorithm and per setting:

In [ ]:
display(CR.membership_table(rates, 'high_score', random=random))
display(CR.membership_table(rates, 'high_score', by_model=True, random=random))

## Every group, per setting

A set in several groups counts in each. Chung–Lu GOAT is where the score's weighting mattered on the saved B: total-degree, harmonic and cut-degree extremes failed (0–3%), while uniqueness (93%) and the score's own extremes (83%) succeeded. `intermediate sets (new)` are this notebook's 40 per pair; `intermediate sets (earlier)` are those in `metric_sets.csv`.

In [ ]:
display(CR.group_table(rates, 'goat', 'chung_lu'))
display(CR.group_table(rates))

## Per pair: does the score still rank sets within the top region?

On the saved B, the score's AUC for success among sets above the random range ranged from 0.07 to well above 0.5. Success rates are less noisy than single runs, so the Spearman correlation between score and success rate among those sets is a cleaner test.

In [ ]:
display(CR.pair_table(rates))

## Success rate against the frozen score

One panel per setting, with all three pairs.
- **Grey dots:** `b_replicates`' random sets.
- **Blue circles:** intermediate sets, earlier and new.
- **Orange diamonds:** metric extremes, pool tails and rules.
- **Aqua triangles:** the lowest-score sets.
- **Dark line:** the mean rate of all sets in equal-width score bins.
- **Dotted line:** the highest random score.

Rates are multiples of 1/11, so dots are jittered up or down by about 0.01.

In [ ]:
CR.plot_overview(rates, random)
plt.show()

## Per setting, one panel per pair

Each constructed set appears once, labelled by its main group: rules first, then the score's own extremes, other metric extremes, pool tails, then intermediate sets. The thin grey lines are 95% Wilson intervals. New intermediate sets are hollow circles in the colour of the metric they head towards (orange: towards the lowest score). At the same score, a gap between colours means the direction of construction matters beyond the score.

In [ ]:
for algorithm in CR.ALGORITHMS:
    for model in CR.MODELS:
        if ((rates['algorithm'] == algorithm) & (rates['model'] == model)).any():
            CR.plot_setting(rates, algorithm, model, random)
            plt.show()

## Save tables and figures

Writes `set_rates.csv`, the tables above as CSV, and PNG/PDF figures to `data/fresh_pair_scores/constructed_redraws/`.

In [ ]:
saved = CR.summarize(save=True)